In [ ]:
# imports

import os
import random
import gradio as gr
from google import genai
from openai import OpenAI
from google.genai import types
from dotenv import load_dotenv
from IPython.display import Markdown, display

In [ ]:
load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ. please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

# Gemma4 API Call

In [ ]:
# Initialize the native Google client
# It automatically picks up the GEMINI_API_KEY environment variable
client = genai.Client()

In [ ]:
def call_gemma(s_message, message):
    """system system_instruction""" 
    # 1. User message
    messages = [
       types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]    
        )
    ]

    # 2. Put system rules
    config = types.GenerateContentConfig(
        system_instruction=s_message,
    )
    # 3. Call the model using the native client
    response = client.models.generate_content(
        model='gemma-4-31b-it', # Swap out with your preferred Gemma 4 variant
        contents=messages,
        config=config
    )

    return response.text

In [ ]:
# call_gemma(system_message, 'Hi There')

In [ ]:
# Description and function declaration the tool for Gemma

ticket_price_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_ticket_price",
            description="Get the airline ticket price for a city",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "destination_city": {
                        "type": "STRING",
                        "description": "Destination city name"
                    }
                },
                "required": ["destination_city"]
            }
        )
    ]
)

 # More Than a City and add or update a ticket price
1. Cities can be asked by one a more at same time.
2. City and price can be add or update.
3. LLM only find function register in AVAILABLE_FUNCTIONS in chat function. Function are easy added on it
4. Chat function follos rule for Google GenAI SDK pattern, FunctionDeclaration is not required because the SDK automatically infers and creates the schema.

In [ ]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_prices(destination_cities: list[str]):
    """
    Returns airline ticket prices for the requested cities.
    Declare type of function's argument for avoiding errors
    """
    if isinstance(destination_cities, str):
        destination_cities = [destination_cities]
        
    return {
        "prices": [
            {
                "city": city,
                "price": ticket_prices.get(
                    city.lower(),
                    "Unknown ticket price"
                )
            }
            for city in destination_cities
        ]
    }

In [ ]:
system_message = """
You are a travel assistant.

Use get_ticket_prices when the user asks for ticket prices.

Use set_price_ticket when the user asks to:
- create a ticket price
- update a ticket price
- change a ticket price

Always pass city names in lowercase.
"""

def set_price_ticket(city: str, price: float):
    """Creates or updates a city ticket price."""

    city = city.lower().strip()

    existed = city in ticket_prices

    ticket_prices[city] = f"${price}"

    return {
        "success": True,
        "action": "updated" if existed else "created",
        "city": city,
        "price": f"${price}",
        "all_prices": ticket_prices
    }

In [ ]:
def chat(message, history):
    # Convert history to GenAI format
    contents = []

    for item in history:
        role = item["role"]

        # Skip system messages because Gemma expects them
        # in system_instruction
        if role == "system":
            continue

        # Map OpenAI roles to GenAI roles
        if role == "assistant":
            genai_role = "model"
        else:
            genai_role = "user"

        contents.append(
            types.Content(
                role=genai_role,
                parts=[types.Part.from_text(text=item["content"])]
            )
        )

    # Add current user message
    contents.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    )

    relevant_system_message = system_message
    if 'belt' in message.lower(): # example
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    config = types.GenerateContentConfig(
        system_instruction=system_message,
        tools=[
            get_ticket_prices,
            set_price_ticket
            ]
    )

    # First call to Gemma
    response = client.models.generate_content(
        model="gemma-4-31b-it",
        contents=contents,
        config=config,
    )

    # Check for function call
    AVAILABLE_FUNCTIONS = {
        "get_ticket_prices": get_ticket_prices,
        "set_price_ticket": set_price_ticket,
    }

    if response.candidates:
        for part in response.candidates[0].content.parts:

            if not part.function_call:
                continue

            function_name = part.function_call.name

            function_to_call = AVAILABLE_FUNCTIONS.get(function_name)

            if function_to_call is None:
                return f"Unknown tool requested: {function_name}"

            tool_result = function_to_call(
                    **dict(part.function_call.args)
                )

            contents.append(response.candidates[0].content)

            contents.append(
                types.Content(
                    role="tool",
                    parts=[
                        types.Part.from_function_response(
                            name=function_name,
                            response=tool_result
                        )
                    ]
                )
            )

            final_response = client.models.generate_content(
                model="gemma-4-31b-it",
                contents=contents,
                config=config,
            )

            return final_response.text

    return response.text

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()